# Emotion-Aware Public Speaking Coach Using Local Large Language Models
RECPAD2026

## Experimental Evaluation

We evaluate the generated feedback using both Likert scale ratings and pairwise preferences, both compared with results from human raters. Section 1 analyses persona, presentation type and judge effects while Section 2 analyses pairwise preferences from LLM-Judges with overall preferences retrieved from human evaluations.

### Load LLM-Judges and Human Ratings

In [2]:
import json
import pandas as pd


## function to process results from scalar and pairwise evaluations into two separate dataframes
def process_results_likert(file_path):
    all_rows = []
    pairwise_rows = []

    comparison_labels = [
        ("academic_simple", "academic_detailed"), # Index 0
        ("pitch_simple", "pitch_detailed"),       # Index 1
        ("academic_simple", "pitch_simple"),       # Index 2
        ("academic_detailed", "pitch_detailed")    # Index 3
    ]
    
    with open(file_path, "r") as f:
        for line in f:
            record = json.loads(line)

            # record contains: index, event, title, category, scalar, pairwise
            # each 'scalar' entry has 4 conditions: academic_simple, academic_detailed, etc.
            for cond_name, metrics in record["scalar"].items():
        
                if metrics:
                    # split condition name (e.g., 'academic_simple') into persona and complexity
                    persona, complexity = cond_name.split("_")
                    
                    all_rows.append({
                        'presentation_id': record["index"], # using index as ID
                        'type': record["category"],        # presentation type
                        'persona': persona,
                        'complexity': complexity,
                        'grounding': metrics.get('grounding_score'),
                        'actionability': metrics.get('actionability_score'),
                        #'persona_consistency': metrics.get('persona_score'),
                        #'clarity': metrics.get('clarity_score'),
                        'overall': metrics.get('overall_score'),
                        #'explanation': metrics.get('explanation')
                    })

          
    
    return pd.DataFrame(all_rows)



gpt41_file_path = "../data/llm_judge_results_gpt41_discrete_3m.jsonl"
gpt4o_file_path = "../data/llm_judge_results_gpt4o_discrete_3m.jsonl"
# load and transform
df_scale = process_results_likert(gpt41_file_path)
df_scale4o = process_results_likert(gpt4o_file_path)
df_scale['avg_score'] = df_scale[['grounding', 'actionability', 'overall']].mean(axis=1)
df_scale4o['avg_score'] = df_scale4o[['grounding', 'actionability', 'overall']].mean(axis=1)
df_scale['condition'] = df_scale['persona'] + "_" + df_scale['complexity']
df_scale4o['condition'] = df_scale4o['persona'] + "_" + df_scale4o['complexity']

df_scale

,presentation_id,type,persona,complexity,grounding,actionability,overall,avg_score,condition
0,0,academic,academic,simple,2,3,2,2.333333,academic_simple
1,0,academic,academic,detailed,3,4,3,3.333333,academic_detailed
2,0,academic,pitch,simple,3,4,3,3.333333,pitch_simple
3,0,academic,pitch,detailed,4,4,4,4.000000,pitch_detailed
4,1,academic,academic,simple,2,3,2,2.333333,academic_simple
...,...,...,...,...,...,...,...,...,...
91,22,pitch,pitch,detailed,5,5,5,5.000000,pitch_detailed
92,23,pitch,academic,simple,5,5,5,5.000000,academic_simple
93,23,pitch,academic,detailed,2,3,2,2.333333,academic_detailed
94,23,pitch,pitch,simple,2,3,2,2.333333,pitch_simple


In [36]:
import pandas as pd

r1 = pd.read_csv("../data/ratings_F_28.csv")
r2 = pd.read_csv("../data/ratings_F_63.csv")
r3 = pd.read_csv("../data/ratings_M_30.csv")
r4 = pd.read_csv("../data/ratings_M_32.csv")

r1["annotator"] = "F_28"
r2["annotator"] = "F_63"
r3["annotator"] = "M_30"
r4["annotator"] = "M_32"

df_human = pd.concat([r1, r2, r3, r4], ignore_index=True)

human_long = df_human.melt(
    id_vars=["n_record", "annotator"],
    value_vars=[
        "academic_simple",
        "academic_detailed",
        "pitch_simple",
        "pitch_detailed"
    ],
    var_name="condition",
    value_name="human_score"
)

human_long.rename(
    columns={"n_record": "presentation_id"},
    inplace=True
)

human_long["item"] = (
    human_long["presentation_id"].astype(str)
    + "_"
    + human_long["condition"]
)

df_human_long = df_human.melt(
    id_vars=["n_record", "annotator"],
    value_vars=[
        "academic_simple",
        "academic_detailed",
        "pitch_simple",
        "pitch_detailed"
    ],
    var_name="condition",
    value_name="overall"
)

df_human_long[["persona", "complexity"]] = (
    df_human_long["condition"]
    .str.extract(r"^(academic|pitch)_(simple|detailed)$")
)

# rename n_record to presentation_id to match the judges
df_human_long = df_human_long.rename(
    columns={"n_record": "presentation_id"}
)

# add type to human observations
presentation_types = (
    df_scale[["presentation_id", "type"]]
    .drop_duplicates()
    .set_index("presentation_id")["type"]
    .to_dict()
)

df_human_long["type"] = (
    df_human_long["presentation_id"]
    .map(presentation_types)
)


df_human_long

,presentation_id,annotator,condition,overall,persona,complexity,type
0,0,F_28,academic_simple,3,academic,simple,academic
1,1,F_28,academic_simple,3,academic,simple,academic
2,2,F_28,academic_simple,4,academic,simple,academic
3,3,F_28,academic_simple,3,academic,simple,academic
4,4,F_28,academic_simple,5,academic,simple,academic
...,...,...,...,...,...,...,...
379,19,M_32,pitch_detailed,3,pitch,detailed,pitch
380,20,M_32,pitch_detailed,3,pitch,detailed,pitch
381,21,M_32,pitch_detailed,3,pitch,detailed,pitch
382,22,M_32,pitch_detailed,3,pitch,detailed,pitch


### 1) Persona, Presentation Type, and Judge Effects
We used linear mixed-effects models to test the effects of persona, presentation type, and judge while accounting for repeated observations from the same presentation, given by the following equation: 
$$Score = Persona \times Type + Judge$$
This formulation models the systematic mean differences between judges.

Presentation type significantly influenced the scores, while the significant Persona $\times$ Type interaction indicates that the relative effect of the two personas differed across presentation types. In addition, GPT-4o assigned systematically lower scores than GPT-4.1, revealing a significant difference in score calibration between the two LLM judges.


In [ ]:
import statsmodels.formula.api as smf

df_41 = df_scale.copy()
df_41["judge"] = "GPT-4.1"

df_4o = df_scale4o.copy()
df_4o["judge"] = "GPT-4o"

df_combined = pd.concat([df_41, df_4o], ignore_index=True)

model1 = smf.mixedlm(
    "overall ~ persona * type + judge", 
    data=df_combined, 
    groups=df_combined["presentation_id"]
).fit()

print(model1.summary())

                  Mixed Linear Model Regression Results
Model:                  MixedLM       Dependent Variable:       overall  
No. Observations:       192           Method:                   REML     
No. Groups:             24            Scale:                    0.7155   
Min. group size:        8             Log-Likelihood:           -246.6421
Max. group size:        8             Converged:                Yes      
Mean group size:        8.0                                              
-------------------------------------------------------------------------
                               Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-------------------------------------------------------------------------
Intercept                       3.210    0.126 25.407 0.000  2.963  3.458
persona[T.pitch]                0.044    0.145  0.304 0.761 -0.240  0.328
type[T.pitch]                   0.653    0.205  3.189 0.001  0.252  1.055
judge[T.GPT-4o]                -0.656    0.122 -5.375 0.

Additionally, actionability was significantly higher for startup pitches than academic presentations ($+0.393$, $p=.011$) and grounding revealed a significant Persona $\times$ Type interaction ($-0.641$, $p=.016$), indicating that persona effectiveness depended on presentation type: the Pitch Mentor achieved slightly higher grounding for academic presentations, whereas the Academic Advisor achieved substantially higher grounding for startup pitches.

In [ ]:
model2 = smf.mixedlm(
    "actionability ~ persona * type", # + judge did not converge
    data=df_combined, 
    groups=df_combined["presentation_id"]
).fit()

print(model2.summary())

                  Mixed Linear Model Regression Results
Model:                  MixedLM     Dependent Variable:     actionability
No. Observations:       192         Method:                 REML         
No. Groups:             24          Scale:                  0.4727       
Min. group size:        8           Log-Likelihood:         -203.8830    
Max. group size:        8           Converged:              Yes          
Mean group size:        8.0                                              
-------------------------------------------------------------------------
                               Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-------------------------------------------------------------------------
Intercept                       3.500    0.083 41.978 0.000  3.337  3.663
persona[T.pitch]                0.059    0.118  0.499 0.618 -0.172  0.290
type[T.pitch]                   0.393    0.154  2.545 0.011  0.090  0.695
persona[T.pitch]:type[T.pitch] -0.309    0.218 -1.414 0.

/Users/sofiafernandes/miniconda3/envs/repos/lib/python3.13/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


In [13]:
model3 = smf.mixedlm(
    "grounding ~ persona * type + judge", 
    data=df_combined, 
    groups=df_combined["presentation_id"]
).fit()

print(model3.summary())

                  Mixed Linear Model Regression Results
Model:                  MixedLM       Dependent Variable:       grounding
No. Observations:       192           Method:                   REML     
No. Groups:             24            Scale:                    0.7048   
Min. group size:        8             Log-Likelihood:           -245.4295
Max. group size:        8             Converged:                Yes      
Mean group size:        8.0                                              
-------------------------------------------------------------------------
                               Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-------------------------------------------------------------------------
Intercept                       2.919    0.126 23.185 0.000  2.673  3.166
persona[T.pitch]                0.176    0.144  1.226 0.220 -0.106  0.459
type[T.pitch]                   0.723    0.204  3.536 0.000  0.322  1.123
judge[T.GPT-4o]                -0.927    0.121 -7.651 0.

Human--LLM correspondence was assessed using Spearman's rank correlation ($\rho$). Human inter-rater reliability was measured using the Intraclass Correlation Coefficient (ICC). Human annotators demonstrated great consistency (\(\mathrm{ICC}(A,k) = 0.746\)), but correspondence with LLM scalar scores was weak: GPT-4.1 showed no significant correlation with the human consensus ($\rho=-0.151$, $p=.142$), while GPT-4o showed a weak positive correlation ($\rho=0.242$, $p=.018$).

In [ ]:
import pingouin as pg
from scipy.stats import pearsonr, spearmanr


# ICC between the four annotators

icc = pg.intraclass_corr(
    data=human_long,
    targets="item",
    raters="annotator",
    ratings="human_score"
)

print("\n========== ICC ==========\n")
print(icc)

# Average-measures absolute agreement ICC
print(
    "\nICC(A,k):"
)

print(
    icc[
        icc["Type"]=="ICC(A,k)"
    ][
        [
            "Type",
            "ICC",
            "CI95",
            "F",
            "pval"
        ]
    ]
)

#-------------
df_human_consensus = (
    df_human_long
    .groupby(
        ["presentation_id", "persona", "complexity", "type"],
        as_index=False
    )["overall"]
    .mean()
    .rename(columns={"overall": "human_overall"})
)

df_agreement = df_combined.merge(
    df_human_consensus,
    on=["presentation_id", "persona", "complexity", "type"],
    how="inner"
)

#print(df_agreement.shape)
#print(df_agreement["judge"].value_counts())

for judge in ["GPT-4.1", "GPT-4o"]:
    
    df_j = df_agreement[df_agreement["judge"] == judge]
    
    rho, p = spearmanr(
        df_j["human_overall"],
        df_j["overall"]
    )
    
    print(f"{judge}:")
    print(f"  Spearman rho = {rho:.3f}")
    print(f"  p-value      = {p:.4f}")


========== ICC ==========

       Type       ICC         F  df1  df2          pval          CI95
0  ICC(1,1)  0.386275  3.517581   95  288  1.668863e-16   [0.28, 0.5]
1  ICC(A,1)  0.422762  5.981972   95  285  4.227882e-32  [0.22, 0.59]
2  ICC(C,1)  0.554664  5.981972   95  285  4.227882e-32  [0.46, 0.65]
3  ICC(1,k)  0.715714  3.517581   95  288  1.668863e-16   [0.61, 0.8]
4  ICC(A,k)  0.745518  5.981972   95  285  4.227882e-32  [0.53, 0.85]
5  ICC(C,k)  0.832831  5.981972   95  285  4.227882e-32  [0.77, 0.88]
GPT-4.1:
  Spearman rho = -0.151
  p-value      = 0.1419
GPT-4o:
  Spearman rho = 0.242
  p-value      = 0.0175


### 2) Pairwise Preference Analysis
Given the judge-dependent effects observed in the Likert scale analysis, we further evaluated whether direct LLM pairwise preferences aligned with human evaluations.  

For pairwise evaluation, each LLM judge directly selected A, B, or Tie for four predefined response comparisons per presentation. Human preference was obtained by summing the four annotators' overall ratings for each response; the response with the higher total was selected, while equal totals were classified as Tie.

Agreement between LLM and human preferences was assessed using raw agreement and Cohen's $\kappa$.

In [43]:
import json

## function to process results from scalar and pairwise evaluations into two separate dataframes
def process_results(file_path):
    all_rows = []
    pairwise_rows = []

    comparison_labels = [
        ("academic_simple", "academic_detailed"), # Index 0
        ("pitch_simple", "pitch_detailed"),       # Index 1
        ("academic_simple", "pitch_simple"),       # Index 2
        ("academic_detailed", "pitch_detailed")    # Index 3
    ]
    
    with open(file_path, "r") as f:
        for line in f:
            record = json.loads(line)

            # record contains: index, event, title, category, scalar, pairwise
            # each 'scalar' entry has 4 conditions: academic_simple, academic_detailed, etc.
            for cond_name, metrics in record["scalar"].items():
        
                if metrics:
                    # split condition name (e.g., 'academic_simple') into persona and complexity
                    persona, complexity = cond_name.split("_")
                    
                    all_rows.append({
                        'presentation_id': record["index"], # using index as ID
                        'type': record["category"],        # presentation type
                        'persona': persona,
                        'complexity': complexity,
                        'grounding': metrics.get('grounding'),
                        'actionability': metrics.get('actionability'),
                        'overall': metrics.get('overall'),
                    })

            for i, res in enumerate(record["pairwise"]):
                if res and i < len(comparison_labels):
                    cond_a, cond_b = comparison_labels[i]
                    pairwise_rows.append({
                        'presentation_id': record["index"],
                        'presentation_type': record["category"],
                        'comparison': f"{cond_a}_vs_{cond_b}",
                        'model_a': cond_a,
                        'model_b': cond_b,
                        'better_grounding': res.get('better_grounding'),
                        'better_actionability': res.get('better_actionability'),
                        'better_overall': res.get('better_overall'),
                
                    })
    
    return pd.DataFrame(all_rows), pd.DataFrame(pairwise_rows)



# load and transform
_, df_pair = process_results("../data/llm_judge_results_gpt41.jsonl")
_, df_pair4o = process_results("../data/llm_judge_results_gpt4o.jsonl")


In [ ]:
from itertools import combinations
import pandas as pd

df_human_consensus = (
    df_human_long
    .groupby(
        ["presentation_id", "condition", "persona", "complexity", "type"],
        as_index=False
    )["overall"]
    .mean()
    .rename(columns={"overall": "human_overall"})
)

def make_pairwise(df, score_col, evaluator_name):
    
    rows = []
    
    conditions = [
        "academic_simple",
        "academic_detailed",
        "pitch_simple",
        "pitch_detailed"
    ]
    
    for presentation_id, group in df.groupby("presentation_id"):
        
        scores = group.set_index("condition")[score_col].to_dict()
        
        for condition_a, condition_b in combinations(conditions, 2):
            
            score_a = scores[condition_a]
            score_b = scores[condition_b]
            
            if score_a > score_b:
                preference = "A"
            elif score_b > score_a:
                preference = "B"
            else:
                preference = "Tie"
            
            rows.append({
                "presentation_id": presentation_id,
                "condition_a": condition_a,
                "condition_b": condition_b,
                "score_a": score_a,
                "score_b": score_b,
                "preference": preference,
                "evaluator": evaluator_name
            })
    
    return pd.DataFrame(rows)



df_human_pairs = make_pairwise(
    df_human_consensus,
    "human_overall",
    "Human"
)

print(df_human_pairs.head())
print(df_human_pairs.shape)



   presentation_id        condition_a        condition_b  score_a  score_b  \
0                0    academic_simple  academic_detailed     4.25      3.0   
1                0    academic_simple       pitch_simple     4.25      4.5   
2                0    academic_simple     pitch_detailed     4.25      3.0   
3                0  academic_detailed       pitch_simple     3.00      4.5   
4                0  academic_detailed     pitch_detailed     3.00      3.0   

  preference evaluator  
0          A     Human  
1          B     Human  
2          A     Human  
3          B     Human  
4        Tie     Human  
(144, 7)


In [50]:
from sklearn.metrics import cohen_kappa_score

df_41 = df_combined[
    df_combined["judge"] == "GPT-4.1"
]

df_41_pairs = make_pairwise(
    df_41,
    "overall",
    "GPT-4.1"
)

df_4o = df_combined[
    df_combined["judge"] == "GPT-4o"
]

df_4o_pairs = make_pairwise(
    df_4o,
    "overall",
    "GPT-4o"
)

df_pairwise_41 = df_human_pairs.merge(
    df_41_pairs,
    on=[
        "presentation_id",
        "condition_a",
        "condition_b"
    ],
    suffixes=("_human", "_llm")
)

df_pairwise_4o = df_human_pairs.merge(
    df_4o_pairs,
    on=[
        "presentation_id",
        "condition_a",
        "condition_b"
    ],
    suffixes=("_human", "_llm")
)


for judge, df in [
    ("GPT-4.1", df_pairwise_41),
    ("GPT-4o", df_pairwise_4o)
]:
    
    kappa = cohen_kappa_score(
        df["preference_human"],
        df["preference_llm"]
    )
    
    print(
        f"{judge}: "
        f"Cohen's kappa = {kappa:.3f}"
    )

complexity_pairs = [
    ("academic_simple", "academic_detailed"),
    ("pitch_simple", "pitch_detailed")
]

persona_pairs = [
    ("academic_simple", "pitch_simple"),
    ("academic_detailed", "pitch_detailed")
]

def pairwise_subset(df, pairs):
    return df[
        df.apply(
            lambda row:
                (row["condition_a"], row["condition_b"]) in pairs,
            axis=1
        )
    ]

df_41_complexity = pairwise_subset(
    df_pairwise_41,
    complexity_pairs
)

df_4o_complexity = pairwise_subset(
    df_pairwise_4o,
    complexity_pairs
)


GPT-4.1: Cohen's kappa = -0.146
GPT-4o: Cohen's kappa = 0.098


In [52]:

# Sum the four human ratings for each presentation × condition
df_human_scores = (
    df_human_long
    .groupby(
        ["presentation_id", "condition"],
        as_index=False
    )["overall"]
    .sum()
    .rename(columns={"overall": "human_score"})
)

print(df_human_scores.head())

comparison_labels = [
    ("academic_simple", "academic_detailed"),
    ("pitch_simple", "pitch_detailed"),
    ("academic_simple", "pitch_simple"),
    ("academic_detailed", "pitch_detailed")
]

def make_human_pairwise(df_human_scores):
    rows = []

    for presentation_id, group in df_human_scores.groupby("presentation_id"):

        scores = (
            group
            .set_index("condition")["human_score"]
            .to_dict()
        )

        for condition_a, condition_b in comparison_labels:

            score_a = scores[condition_a]
            score_b = scores[condition_b]

            if score_a > score_b:
                preference = "A"
            elif score_b > score_a:
                preference = "B"
            else:
                preference = "Tie"

            rows.append({
                "presentation_id": presentation_id,
                "model_a": condition_a,
                "model_b": condition_b,
                "human_score_a": score_a,
                "human_score_b": score_b,
                "preference_human": preference
            })

    return pd.DataFrame(rows)


df_human_pair = make_human_pairwise(df_human_scores)

print(df_human_pair.head())
print(df_human_pair.shape)



   presentation_id          condition  human_score
0                0  academic_detailed           12
1                0    academic_simple           17
2                0     pitch_detailed           12
3                0       pitch_simple           18
4                1  academic_detailed           13
   presentation_id            model_a            model_b  human_score_a  \
0                0    academic_simple  academic_detailed             17   
1                0       pitch_simple     pitch_detailed             18   
2                0    academic_simple       pitch_simple             17   
3                0  academic_detailed     pitch_detailed             12   
4                1    academic_simple  academic_detailed             15   

   human_score_b preference_human  
0             12                A  
1             12                A  
2             18                B  
3             12              Tie  
4             13                A  
(96, 6)


In [53]:
# 1. Print human preference distributions
print(df_human_pair["preference_human"].value_counts())
print(df_human_pair["preference_human"].value_counts(normalize=True).mul(100).round(1))


# 2. Helper function to process pairwise comparison dataframes
def prepare_pairwise_comparison(df_pair_source):
    df_compare = df_human_pair.merge(
        df_pair_source[["presentation_id", "model_a", "model_b", "better_overall"]],
        on=["presentation_id", "model_a", "model_b"],
        how="inner"
    ).rename(columns={"better_overall": "preference_llm"})
    
    print("Shape:", df_compare.shape)
    print("LLM Value Counts:\n", df_compare["preference_llm"].value_counts(dropna=False))
    return df_compare


print("\n--- GPT-4.1 Setup ---")
df_41_compare = prepare_pairwise_comparison(df_pair)

print("\n--- GPT-4o Setup ---")
df_4o_compare = prepare_pairwise_comparison(df_pair4o)


# 3. Evaluation function for agreement, kappa, and crosstabs
def evaluate_pairwise_agreement(df, name):
    agreement = (df["preference_human"] == df["preference_llm"]).mean()
    kappa = cohen_kappa_score(
        df["preference_human"],
        df["preference_llm"],
        labels=["A", "B", "Tie"]
    )

    print(f"\n{name}")
    print("-" * 30)
    print(f"Comparisons: {len(df)}")
    print(f"Agreement:   {agreement:.3%}")
    print(f"Cohen's κ:   {kappa:.3f}")

    print("\nHuman:\n", df["preference_human"].value_counts())
    print("\nLLM:\n", df["preference_llm"].value_counts())

    cm = pd.crosstab(
        df["preference_human"],
        df["preference_llm"],
        rownames=["Human"],
        colnames=[name],
        dropna=False
    )
    print(f"\nConfusion Matrix ({name}):\n", cm)

    return agreement, kappa


# 4. Evaluate both models in a loop
agreement_41, kappa_41 = evaluate_pairwise_agreement(df_41_compare, "GPT-4.1")
agreement_4o, kappa_4o = evaluate_pairwise_agreement(df_4o_compare, "GPT-4o")

preference_human
A      63
B      24
Tie     9
Name: count, dtype: int64
preference_human
A      65.6
B      25.0
Tie     9.4
Name: proportion, dtype: float64

--- GPT-4.1 Setup ---
Shape: (96, 7)
LLM Value Counts:
 preference_llm
A    50
B    46
Name: count, dtype: int64

--- GPT-4o Setup ---
Shape: (96, 7)
LLM Value Counts:
 preference_llm
A    67
B    29
Name: count, dtype: int64

GPT-4.1
------------------------------
Comparisons: 96
Agreement:   68.750%
Cohen's κ:   0.420

Human:
 preference_human
A      63
B      24
Tie     9
Name: count, dtype: int64

LLM:
 preference_llm
A    50
B    46
Name: count, dtype: int64

Confusion Matrix (GPT-4.1):
 GPT-4.1   A   B
Human          
A        45  18
B         3  21
Tie       2   7

GPT-4o
------------------------------
Comparisons: 96
Agreement:   64.583%
Cohen's κ:   0.241

Human:
 preference_human
A      63
B      24
Tie     9
Name: count, dtype: int64

LLM:
 preference_llm
A    67
B    29
Name: count, dtype: int64

Confusion Matrix (GP

In [54]:
def evaluate_pairwise_agreement(df, name):

    agreement = (
        df["preference_human"] == df["preference_llm"]
    ).mean()

    kappa = cohen_kappa_score(
        df["preference_human"],
        df["preference_llm"],
        labels=["A", "B", "Tie"]
    )

    print(f"\n{name}")
    print("-" * 30)
    print(f"Comparisons: {len(df)}")
    print(f"Agreement:   {agreement:.3%}")
    print(f"Cohen's κ:   {kappa:.3f}")

    print("\nHuman:")
    print(df["preference_human"].value_counts())

    print("\nLLM:")
    print(df["preference_llm"].value_counts())

    return agreement, kappa


agreement_41, kappa_41 = evaluate_pairwise_agreement(
    df_41_compare,
    "GPT-4.1"
)

agreement_4o, kappa_4o = evaluate_pairwise_agreement(
    df_4o_compare,
    "GPT-4o"
)


GPT-4.1
------------------------------
Comparisons: 96
Agreement:   68.750%
Cohen's κ:   0.420

Human:
preference_human
A      63
B      24
Tie     9
Name: count, dtype: int64

LLM:
preference_llm
A    50
B    46
Name: count, dtype: int64

GPT-4o
------------------------------
Comparisons: 96
Agreement:   64.583%
Cohen's κ:   0.241

Human:
preference_human
A      63
B      24
Tie     9
Name: count, dtype: int64

LLM:
preference_llm
A    67
B    29
Name: count, dtype: int64


In [55]:
for name, df in [
    ("GPT-4.1", df_41_compare),
    ("GPT-4o", df_4o_compare)
]:

    cm = pd.crosstab(
        df["preference_human"],
        df["preference_llm"],
        rownames=["Human"],
        colnames=[name],
        dropna=False
    )

    print(f"\n{name}")
    print(cm)


GPT-4.1
GPT-4.1   A   B
Human          
A        45  18
B         3  21
Tie       2   7

GPT-4o
GPT-4o   A   B
Human         
A       50  13
B       12  12
Tie      5   4


In [56]:
from sklearn.metrics import cohen_kappa_score

for name, df in [
    ("GPT-4.1", df_41_compare),
    ("GPT-4o", df_4o_compare)
]:

    # Exclude cases where humans were tied
    df_clear = df[
        df["preference_human"].isin(["A", "B"])
    ].copy()

    agreement_clear = (
        df_clear["preference_human"]
        == df_clear["preference_llm"]
    ).mean()

    kappa_clear = cohen_kappa_score(
        df_clear["preference_human"],
        df_clear["preference_llm"],
        labels=["A", "B"]
    )

    print(f"\n{name}")
    print("-" * 30)
    print(f"Clear human comparisons: {len(df_clear)}")
    print(f"Agreement: {agreement_clear:.3%}")
    print(f"Cohen's κ: {kappa_clear:.3f}")


GPT-4.1
------------------------------
Clear human comparisons: 87
Agreement: 75.862%
Cohen's κ: 0.494

GPT-4o
------------------------------
Clear human comparisons: 87
Agreement: 71.264%
Cohen's κ: 0.290
